In [1]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.cluster import KMeans

c:\Users\danni\anaconda3\envs\psi\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1055.18it/s]


In [2]:
aliceData = []
with open("Datasets/ENCRYPTO/emails_alice.txt", "r") as f:
    for line in f:
        aliceData.append(line.strip())

In [3]:
bobData = []
with open("Datasets/ENCRYPTO/emails_bob.txt", "r") as f:
    for line in f:
        bobData.append(line.strip())

In [6]:
embeddingAlice = model.encode(aliceData)
embeddingBob = model.encode(bobData)

In [7]:
n, k = 384, 2

rng = np.random.default_rng(42)
R = rng.standard_normal((n, k)) / np.sqrt(k)

alice_projected = embeddingAlice @ R
bob_projected = embeddingBob@R

In [ ]:
numAliceClusters = 20
KMeansAlice = KMeans(numAliceClusters)
AliceLabels = KMeansAlice.fit_predict(alice_projected)

numBobClusters = 30
KMeansBob = KMeans(numBobClusters)
BobLabels = KMeansBob.fit_predict(bob_projected)

In [19]:
AliceDistances = np.linalg.norm(
    alice_projected - KMeansAlice.cluster_centers_[AliceLabels], axis=1
)

radiiAlice = np.array([
    AliceDistances[AliceLabels == i].max()
    for i in range(KMeansAlice.n_clusters)
])

print(radiiAlice)

[0.27763973 0.53799932 0.50593065 0.36623496 0.29895169 0.75938201
 0.32979049 0.58787849 0.53212337 0.29445952 0.63270933 0.2355999
 0.993172   0.27423605 0.29105235 0.63601291 0.29728878 0.53393224
 0.49406108 0.2808857 ]


In [22]:
BobDistances = np.linalg.norm(
    bob_projected - KMeansBob.cluster_centers_[BobLabels], axis=1
)

radiiBob = np.array([
    BobDistances[BobLabels == i].max()
    for i in range(KMeansBob.n_clusters)
])

print(radiiBob)

[0.2956801  0.18732911 0.2398559  0.38742477 0.27476711 0.20327622
 0.34828927 0.3033831  0.63695477 0.24363339 0.32743639 0.41203195
 0.24441582 0.40659544 0.23258451 0.48537782 0.19477924 0.49561792
 0.77679235 0.22565623 0.23771688 0.32977996 0.21025905 0.35529353
 0.2152664  0.37869285 0.25259242 0.21398834 0.37990353 0.2332362 ]


In [28]:
def get_cluster_region(centers, i):
    """
    Return A, b defining Voronoi region i:

        A @ x <= b
    """
    c = centers[i]

    A = 2 * (centers - c)
    b = (
        np.sum(centers**2, axis=1)
        - np.sum(c**2)
    )

    return A, b

In [32]:
import numpy as np
from scipy.optimize import linprog

def voronoi_constraints(centers, i):
    c = centers[i]
    others = np.delete(centers, i, axis=0)

    diff = others - c
    norms = np.linalg.norm(diff, axis=1)

    if np.any(norms == 0):
        raise ValueError("Duplicate centroids")

    A = diff / norms[:, None]
    b = np.sum(A * (others + c) / 2, axis=1)

    return A, b


def find_overlapping_regions(centers_a, centers_b, tol=1e-8):
    centers_a = np.asarray(centers_a, dtype=float)
    centers_b = np.asarray(centers_b, dtype=float)

    if centers_a.ndim != 2 or centers_b.ndim != 2:
        raise ValueError("Expected 2D centroid arrays")
    if centers_a.shape[1] != centers_b.shape[1]:
        raise ValueError("Vector dimensions must match")

    d = centers_a.shape[1]

    constraints_a = [
        voronoi_constraints(centers_a, i)
        for i in range(len(centers_a))
    ]
    constraints_b = [
        voronoi_constraints(centers_b, j)
        for j in range(len(centers_b))
    ]

    overlaps = []

    for i, (A1, b1) in enumerate(constraints_a):
        for j, (A2, b2) in enumerate(constraints_b):

            A = np.vstack([A1, A2])
            b = np.concatenate([b1, b2])

            # Find an interior point by maximizing
            # common slack t in A @ x + t <= b.
            result = linprog(
                c=np.r_[np.zeros(d), -1.0],
                A_ub=np.column_stack([A, np.ones(len(b))]),
                b_ub=b,
                bounds=[(None, None)] * d + [(0, 1)],
                method="highs",
            )

            if result.status == 0:
                if result.x[-1] > tol:
                    overlaps.append((i, j))
            elif result.status != 2:
                raise RuntimeError(result.message)

    return np.array(overlaps, dtype=int).reshape(-1, 2)

In [ ]:
AliceRegions = {}
for i in range(20):
    

In [33]:
A, b = get_cluster_region(
    KMeansAlice.cluster_centers_, i=0
)
print(A)
print(b)

[[ 0.          0.        ]
 [-0.38506983 -2.00486678]
 [-1.87211233  0.18197274]
 [ 1.28758975 -0.78789882]
 [-0.00443301  0.74723813]
 [-1.39291013 -1.5251299 ]
 [ 0.63393792  1.05769071]
 [ 0.64819895  1.98490929]
 [ 1.69658385  0.71933714]
 [-0.92906887 -0.25717152]
 [-1.40517232  0.83083803]
 [ 0.36282494 -0.53930543]
 [ 0.7846367  -2.34167845]
 [ 0.47641653 -1.37743234]
 [-0.39039209 -0.98289737]
 [ 2.16731508 -0.77193403]
 [ 0.95249104  0.08341607]
 [-0.77184327  1.42882004]
 [-1.64659494 -0.68977979]
 [-0.60812508  0.32294799]]
[0.         0.93198408 0.62988844 0.72608    0.160053   0.83030489
 0.49792107 1.23590676 1.10461705 0.09618235 0.49468659 0.14075043
 1.56759818 0.55832443 0.19774584 1.60219291 0.36304311 0.59252973
 0.54887998 0.04326821]


In [ ]:
AliceCenters = KMeansAlice